# 🎙️ Bengali ASR & Speaker Diarization - Web App

This notebook runs the FastAPI web application on Google Colab with GPU.
It uses ngrok to create a public URL so you can access it from anywhere.

## Steps:
1. Run Cell 1 to install dependencies
2. Run Cell 2 to set up ngrok (get free token from https://ngrok.com)
3. Run Cell 3 to start the server
4. Click the ngrok URL to use the app!

In [ ]:
#@title 1️⃣ Install Dependencies
!pip install -q fastapi uvicorn[standard] python-multipart aiofiles
!pip install -q faster-whisper torch torchaudio
!pip install -q soundfile librosa noisereduce bnunicodenormalizer silero-vad
!pip install -q huggingface-hub toml pyngrok

# Install DiariZen
import os
if not os.path.exists('/content/DiariZen'):
    !git clone https://github.com/BUTSpeechFIT/DiariZen.git /content/DiariZen
!pip install -q -e /content/DiariZen/pyannote-audio
!pip install -q -e /content/DiariZen

# Install ffmpeg
!apt-get install -y -qq ffmpeg > /dev/null 2>&1

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('✅ All dependencies installed!')

In [ ]:
#@title 2️⃣ Setup App Files
import os

APP_DIR = '/content/webapp'
os.makedirs(APP_DIR, exist_ok=True)
os.makedirs(f'{APP_DIR}/static', exist_ok=True)
os.makedirs(f'{APP_DIR}/uploads', exist_ok=True)
os.makedirs(f'{APP_DIR}/results', exist_ok=True)

# ── If you uploaded the webapp folder to Colab or mounted Google Drive, ──
# ── copy from there. Otherwise, we write the files inline below.        ──

# Option A: Clone from your repo (if you pushed to GitHub)
# !git clone https://github.com/YOUR_USER/YOUR_REPO.git /content/webapp

# Option B: Upload from local machine
# Use the Colab file browser to upload config.py, pipeline.py, app.py, static/index.html
# to /content/webapp/

# Option C: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/webapp/* /content/webapp/

print('📁 App directory structure:')
for root, dirs, files in os.walk(APP_DIR):
    level = root.replace(APP_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')

In [ ]:
#@title 3️⃣ Configure Model Paths
import os

# ── ASR Model ──
# Option 1: HuggingFace repo ID (will download automatically)
os.environ['ASR_MODEL_PATH'] = 'faysal314/whisper-md-lora-ep7-ct2'

# Option 2: If you have the CT2 model on Kaggle/Drive, use local path:
# os.environ['ASR_MODEL_PATH'] = '/content/drive/MyDrive/models/whisper_md_lora_e7_ct2'

# ── Diarization Model ──
os.environ['DIARIZEN_REPO_ID'] = 'BUT-FIT/diarizen-wavlm-large-s80-md-v2'

# Fine-tuned weights (optional - set path if you have them)
# os.environ['DIARIZEN_FINETUNED_PATH'] = '/content/drive/MyDrive/models/best_diarizen_retrained.pt'

# ── HuggingFace Token (needed for pyannote models) ──
# Get token from https://huggingface.co/settings/tokens
os.environ['HF_TOKEN'] = ''  # ← PUT YOUR HF TOKEN HERE

# ── Compute settings ──
os.environ['ASR_COMPUTE_TYPE'] = 'float16'  # 'float16' for GPU, 'int8' for CPU
os.environ['ASR_DEVICE'] = 'auto'

print('✅ Environment configured')
print(f"   ASR Model: {os.environ['ASR_MODEL_PATH']}")
print(f"   Diarization: {os.environ['DIARIZEN_REPO_ID']}")

In [ ]:
#@title 4️⃣ Start Web Server with ngrok
#@markdown Get your free ngrok auth token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = ''  #@param {type:"string"}

import os, sys, subprocess, threading, time

# Setup ngrok
from pyngrok import ngrok, conf
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
else:
    print('⚠️  No ngrok token set. Get one free at https://ngrok.com')
    print('   Without it, the tunnel may not work.')

# Start the FastAPI server in background
APP_DIR = '/content/webapp'
os.chdir(APP_DIR)
sys.path.insert(0, APP_DIR)

def run_server():
    import uvicorn
    uvicorn.run('app:app', host='0.0.0.0', port=7860, log_level='info')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)  # Wait for server to start

# Create ngrok tunnel
public_url = ngrok.connect(7860)
print('=' * 60)
print(f'🌐 Your app is live at: {public_url}')
print('=' * 60)
print('\nOpen this URL in your browser to use the app.')
print('The server will run as long as this cell is running.')
print('\nPress the stop button ⬛ to shut down.')

# Keep alive
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    ngrok.disconnect(public_url)
    print('\n🛑 Server stopped.')